# 蓖麻油粘滞系数计算 (斯托克斯定律)

本项目用于计算落球法测量蓖麻油粘滞系数实验的数据处理与不确定度评定。

---
### 实验原理与计算公式

#### 1. 粘滞系数计算公式（考虑管壁修正）
小球在液体中受重力、浮力与斯托克斯粘滞阻力作用达到终端匀速下落速度 $v = l/t$。考虑到圆筒玻璃管壁的有限边界效应，引入管壁修正因子 $(1 + 2.4 d/D)$：
$$\eta = \frac{(\rho - \rho_0) g d^2 t}{18 l \left(1 + 2.4 \frac{d}{D}\right)}$$

* $\rho$：小球密度（钢球通常取 $7800\,\mathrm{kg/m^3}$）
* $\rho_0$：液体（蓖麻油）密度（室温下约 $960\,\mathrm{kg/m^3}$）
* $g$：重力加速度（取 $9.80\,\mathrm{m/s^2}$）
* $d$：小球直径
* $D$：玻璃管内径
* $l$：小球下落距离
* $t$：下落平均时间

#### 2. 雷诺数（$Re$）判断与奥辛（Oseen）修正
斯托克斯公式成立的条件为层流状态且惯性力远小于粘性阻力，其雷诺数定义为：
$$Re = \frac{\rho_0 v d}{\eta}$$
* 当 $Re \le 0.1$ 时，斯托克斯定律完全适用，无需修正。
* 当 $0.1 < Re \le 1.0$ 时，流场出现微小扰动，采用奥辛公式进行一级修正：
$$\eta_{\mathrm{corrected}} = \frac{\eta}{1 + \frac{3}{16} Re}$$

#### 3. 不确定度传递公式
假定 $\rho, \rho_0, g$ 的误差可忽略不计，由误差合成法则，粘滞系数的相对不确定度传递公式为：
$$u_r(\eta) = \frac{u_\eta}{\eta} = \sqrt{ \left(2 \frac{u_d}{d}\right)^2 + \left(\frac{u_t}{t}\right)^2 + \left(\frac{u_l}{l}\right)^2 + \left(\frac{u_D}{D}\right)^2 }$$

绝对不确定度为：
$$u_\eta = \eta \cdot u_r(\eta)$$

其中各物理量 $X$ 的不确定度按 A 类与 B 类合成：
$$u_X = \sqrt{u_{A,X}^2 + u_{B,X}^2}, \quad u_{A,X} = \frac{s_X}{\sqrt{n}}, \quad u_{B,X} = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$$

In [1]:
import math
from decimal import Decimal
from python.utils import scientific_round, calculate_stats

print("工具函数与模块加载完成。")

工具函数与模块加载完成。


### 1. 实验参数与数据输入
> **提示**：可直接在下方单元格中修改小球下落时间测量值 `t_raw_data` 以及各项仪器参数，无需命令行逐次敲入。

In [2]:
# --- 物理常数与材料密度 ---
G = Decimal("9.80")          # 重力加速度 (m/s^2)
RHO_BALL = Decimal("7800")   # 小球(钢球)密度 (kg/m^3)
RHO_OIL = Decimal("960")     # 蓖麻油密度 (kg/m^3)

# --- 几何与仪器参数 (单位: mm / cm) ---
D_mean = Decimal("20.0")      # 玻璃管直径 D (mm)
d_mean = Decimal("1.0")       # 小球直径 d (mm)
l_mean = Decimal("20.0")      # 下落距离 l (cm)

# --- 仪器允差 (仪器误差) Δ_inst ---
delta_D = Decimal("0")         # D 的仪器误差 (mm)，如未单独评估可设为 0
delta_d = Decimal("0")         # d 的仪器误差 (mm)，如未单独评估可设为 0
delta_l = Decimal("0")         # l 的仪器误差 (mm)，如未单独评估可设为 0
delta_t = Decimal("0.01")      # 秒表仪器误差 (s)

# --- 5次下落时间测量值 (s) ---
# 请在此填入实际实验数据：
t_raw_data = [Decimal("12.45"), Decimal("12.38"), Decimal("12.50"), Decimal("12.42"), Decimal("12.48")]

print(f"参数设置完成:")
print(f"- 玻璃管直径 D: {D_mean} mm")
print(f"- 小球直径 d:   {d_mean} mm")
print(f"- 下落距离 l:   {l_mean} cm")
print(f"- 时间测量值 t:  {[float(x) for x in t_raw_data]} s")

参数设置完成:
- 玻璃管直径 D: 20.0 mm
- 小球直径 d:   1.0 mm
- 下落距离 l:   20.0 cm
- 时间测量值 t:  [12.45, 12.38, 12.5, 12.42, 12.48] s


### 2. 测量数据统计与不确定度计算

In [3]:
# 长度单位换算为 mm 以便统一不确定度计算
l_mean_mm = l_mean * Decimal("10")

# 单次测量参数的 B 类不确定度 (u = Δ_inst / √3)
D_u = delta_D / Decimal(str(math.sqrt(3)))
d_u = delta_d / Decimal(str(math.sqrt(3)))
l_u = delta_l / Decimal(str(math.sqrt(3)))

# 5次时间测量的统计量 (平均值、合成不确定度、A类、B类)
t_mean, t_u, t_ua, t_ub, t_s = calculate_stats(t_raw_data, delta_t)

print("--- 时间统计分析 ---")
print(f"时间平均值 t_mean : {t_mean:.4f} s")
print(f"样本标准差 s(t)   : {t_s:.4f} s")
print(f"A 类不确定度 u_A(t): {t_ua:.4f} s")
print(f"B 类不确定度 u_B(t): {t_ub:.4f} s")
print(f"合成不确定度 u(t)  : {t_u:.4f} s")

--- 时间统计分析 ---
时间平均值 t_mean : 12.4460 s
样本标准差 s(t)   : 0.0477 s
A 类不确定度 u_A(t): 0.0214 s
B 类不确定度 u_B(t): 0.0058 s
合成不确定度 u(t)  : 0.0221 s


### 3. 粘滞系数计算、雷诺数校验与修约输出

In [4]:
# 统一转换为国际单位制 (m, kg, s)
D_m = D_mean / Decimal("1000")
d_m = d_mean / Decimal("1000")
l_m = l_mean_mm / Decimal("1000")

# --- 1. 斯托克斯粘滞系数计算 (带管壁修正) ---
correction_factor = Decimal("1") + Decimal("2.4") * (d_m / D_m)
numerator = (RHO_BALL - RHO_OIL) * G * (d_m**2) * t_mean
denominator = Decimal("18") * l_m * correction_factor
eta_val = numerator / denominator

# --- 2. 下落终端速度与雷诺数 Re 校验 ---
v_val = l_m / t_mean
Re = RHO_OIL * v_val * d_m / eta_val

# --- 3. 雷诺数判断与 Oseen 修正 ---
if Re > Decimal("0.1"):
    eta_corrected = eta_val / (Decimal("1") + Decimal("3") / Decimal("16") * Re)
    oseen_applied = True
else:
    eta_corrected = eta_val
    oseen_applied = False

# --- 4. 相对不确定度与绝对不确定度 ---
# 各相对不确定度平方和项: (2*ud/d)^2 + (ut/t)^2 + (ul/l)^2 + (uD/D)^2
term_d = (Decimal("2") * d_u / d_mean) if d_mean != 0 else Decimal("0")
term_t = (t_u / t_mean) if t_mean != 0 else Decimal("0")
term_l = (l_u / l_mean_mm) if l_mean_mm != 0 else Decimal("0")
term_D = (D_u / D_mean) if D_mean != 0 else Decimal("0")

rel_u_sq = term_d**2 + term_t**2 + term_l**2 + term_D**2
rel_u = Decimal(str(math.sqrt(float(rel_u_sq))))
eta_u = eta_corrected * rel_u

# --- 5. 按照科学修约规范对齐保留有效数字 ---
eta_final, u_final = scientific_round(eta_corrected, eta_u)

# --- 6. 实验报告格式化输出 ---
print("=" * 45)
print("               实 验 计 算 结 果              ")
print("=" * 45)
print(f"下落平均时间 t    : {t_mean:.3f} ± {t_u:.3f} s")
print(f"小球下落速度 v    : {v_val:.5f} m/s")
print("-" * 45)
print(f"雷诺数 Re         : {Re:.4f}")
if not oseen_applied:
    print("判定结论          : Re <= 0.1，满足斯托克斯层流条件，无需 Oseen 修正。")
else:
    print("判定结论          : Re > 0.1，已自动应用 Oseen 修正公式。")
    print(f"修正前粘滞系数 η0 : {eta_val:.4f} Pa·s")
print("-" * 45)
print(f"相对不确定度 u_r  : {rel_u * 100:.2f}%")
print(f"绝对不确定度 u_η  : {eta_u:.4f} Pa·s")
print("-" * 45)
print(f"最终结果表示      : η = {eta_final} ± {u_final} Pa·s")
print("=" * 45)

               实 验 计 算 结 果              
下落平均时间 t    : 12.446 ± 0.022 s
小球下落速度 v    : 0.01607 m/s
---------------------------------------------
雷诺数 Re         : 0.0746
判定结论          : Re <= 0.1，满足斯托克斯层流条件，无需 Oseen 修正。
---------------------------------------------
相对不确定度 u_r  : 0.18%
绝对不确定度 u_η  : 0.0004 Pa·s
---------------------------------------------
最终结果表示      : η = 0.2069 ± 0.0004 Pa·s
